In [1]:
import os, glob
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

print("Input directories:", os.listdir("../input"))




Input directories: ['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'description.md', 'jigsaw-toxic-comment-classification-challenge', 'sample_submission.csv.zip', 'train.csv', 'test.csv']


In [2]:
train_path = "../input/jigsaw-toxic-comment-classification-challenge/train.csv"
test_path = "../input/jigsaw-toxic-comment-classification-challenge/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]




In [3]:
tfidf = TfidfVectorizer(
    max_features=50000, ngram_range=(1, 2), stop_words="english", dtype=np.float32
)

X_train = tfidf.fit_transform(train_df["comment_text"].fillna(""))
X_test = tfidf.transform(test_df["comment_text"].fillna(""))

base_clf = LogisticRegression(
    C=4.0, max_iter=200, solver="saga", n_jobs=-1, class_weight="balanced"
)
clf = OneVsRestClassifier(base_clf)

clf.fit(X_train, train_df[label_cols])

# predict_proba already returns an (n_samples, n_classes) array
test_preds = clf.predict_proba(X_test)

# Directly create DataFrame with correct shape
pred_df = pd.DataFrame(test_preds, columns=label_cols)




/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which 

In [4]:
submission = pd.concat([test_df[["id"]].reset_index(drop=True), pred_df], axis=1)
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv with shape:", submission.shape)

Submission saved to submission.csv with shape: (153164, 7)
